In [35]:
from langchain_ollama import OllamaEmbeddings
from langchain_ollama import OllamaLLM
from langchain_chroma import Chroma

OLLAMA_URL = "http://10.10.10.100:11434"

EMBED_MODEL = "nomic-embed-text-v2-moe"

LLM_MODEL = (
    "hf.co/ggufbench/"
    "Qwen3.6-27B-4bpw-16GB-VRAM:latest"
)

In [36]:
import requests

URL = (
    "https://services.arcgis.com/"
    "V6ZHFr6zdgNZuVG0/arcgis/rest/services/"
    "California_Highways/FeatureServer/0/query"
)

features = []

offset = 0
batch_size = 2000

while True:

    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": True,
        "f": "json",
        "resultOffset": offset,
        "resultRecordCount": batch_size
    }

    r = requests.get(
        URL,
        params=params,
        timeout=120
    )

    data = r.json()

    batch = data.get(
        "features",
        []
    )

    if not batch:
        break

    features.extend(batch)

    offset += batch_size

    print(
        f"Downloaded {len(features)}"
    )

print(
    f"\nTotal Features: {len(features)}"
)

Downloaded 508

Total Features: 508


In [37]:
routes = sorted(
    {
        str(
            f["attributes"]
            .get("ROUTE")
        )
        for f in features
    }
)

print(
    f"Unique Routes: {len(routes)}"
)

print(routes[:50])

print(
    "\nContains Route 5:",
    "5" in routes
)

Unique Routes: 254
['1', '10', '101', '101U', '103', '104', '105', '107', '108', '109', '10S', '11', '110', '111', '112', '113', '114', '115', '116', '118', '119', '12', '120', '121', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '14', '140', '142', '144', '145', '146', '147', '149']

Contains Route 5: True


In [38]:
import json

from langchain_core.documents import Document

docs = []

for feature in features:

    attrs = feature["attributes"]

    geometry = feature.get(
        "geometry",
        {}
    )

    route = attrs.get("ROUTE")

    direction = attrs.get("DIR")

    text = json.dumps(
        {
            "layer":
                "California_Highways",

            "route":
                route,

            "route_id":
                attrs.get("ROUTE_ID"),

            "direction":
                direction,

            "year":
                attrs.get("YEAR_REC"),

            "state_code":
                attrs.get("STATE_CODE"),

            "comments":
                attrs.get("COMMENTS"),

            "route_geometry_id":
                attrs.get("RTE_GEOMID"),

            "shape_length":
                attrs.get("Shape_Leng"),

            "geometry":
                geometry,

            "description":
                (
                    f"California State Route "
                    f"{route} travelling "
                    f"{direction}bound"
                )
        },
        default=str
    )

    docs.append(
        Document(
            page_content=text,
            metadata={
                "route":
                    str(route),

                "direction":
                    str(direction),

                "objectid":
                    attrs.get(
                        "OBJECTID"
                    )
            }
        )
    )

print(
    f"Documents: {len(docs)}"
)

Documents: 508


In [39]:
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_URL
)

vector = embeddings.embed_query(
    docs[0].page_content
)

print(
    "Embedding Dimension:",
    len(vector)
)

Embedding Dimension: 768


In [40]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="ca_highways_geo"
)

print(
    "Vector DB Built"
)

Vector DB Built


In [41]:
results = vectorstore.similarity_search(
    "Route 5",
    k=5
)

for r in results:

    print(
        r.metadata
    )

    print(
        r.page_content[:500]
    )

    print("=" * 100)

{'route': '5', 'direction': 'N', 'objectid': 406}
{"layer": "California_Highways", "route": "5", "route_id": "5R", "direction": "N", "year": 2015, "state_code": 6, "comments": " ", "route_geometry_id": "5_20180226_R", "shape_length": 0, "geometry": {"paths": [[[-117.03243597584, 32.5444843461757], [-117.033259218838, 32.5450019662678], [-117.03451795404, 32.5458205004158], [-117.035567294695, 32.5464915070775], [-117.036892656171, 32.5473552510441], [-117.038686768579, 32.5484926344276], [-117.039724214801, 32.549182677039], [-117.040582599707,
{'direction': 'N', 'route': '5', 'objectid': 406}
{"layer": "California_Highways", "route": "5", "route_id": "5R", "direction": "N", "year": 2015, "state_code": 6, "comments": " ", "route_geometry_id": "5_20180226_R", "shape_length": 0, "geometry": {"paths": [[[-117.03243597584, 32.5444843461757], [-117.033259218838, 32.5450019662678], [-117.03451795404, 32.5458205004158], [-117.035567294695, 32.5464915070775], [-117.036892656171, 32.54735525104

In [42]:
retriever = (
    vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": 10,
            "fetch_k": 50
        }
    )
)

In [ ]:
from langchain_core.prompts import (
    ChatPromptTemplate
)

from langchain_core.output_parsers import (
    StrOutputParser
)

from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough
)

def format_docs(docs):

    return "\n\n".join(
        d.page_content
        for d in docs
    )

prompt = (
    ChatPromptTemplate
    .from_template(
"""
You are an expert California GIS analyst.

Use only the supplied context.

Answer directly in 3 concise sentences. Do not include reasoning or thinking.

Context:
{context}

Question:
{question}

Answer:
"""
    )
)

chain = (
    {
        "context":
            retriever
            | RunnableLambda(
                format_docs
            ),

        "question":
            RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [50]:
response = chain.invoke(
    "In 3 concise sentences, tell me about Route 5."
)

answer = response.split("</think>", 1)[-1].strip()

if not answer:
    answer = response.strip()

print(answer[:1000])

California State Route 5 is a scenic highway in Northern California that connects the Central Valley to the Pacific Ocean via the Sierra Nevada mountains. It is renowned for its steep grades and winding curves, offering dramatic views of the San Andreas Fault and the Coast Ranges. The route serves as a vital link for travelers moving between the Bay Area and the Central Coast, terminating at Monterey.<|endoftext|><|im_start|>user


In [53]:
response = chain.invoke(
    "In 3 concise sentences, tell me about Interstate 5."
)

answer = response.split("</think>", 1)[-1].strip()

if not answer:
    answer = response.strip()

print(answer[:1000])